In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.utils as vutils
import os
import json
import random
import shutil
import warnings
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Detectron2 Imports
import detectron2
from detectron2.utils.logger import setup_logger
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, default_setup
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_train_loader, build_detection_test_loader
from detectron2.data import detection_utils as utils
from detectron2.structures import BoxMode, Instances, Boxes, BitMasks
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.modeling.backbone import Backbone, BACKBONE_REGISTRY
from detectron2.modeling.backbone.fpn import FPN, LastLevelMaxPool
from detectron2.layers import ShapeSpec
import detectron2.utils.comm as comm
import fvcore.nn.weight_init as weight_init

# Hugging Face & Albumentations
from transformers import ViTMAEForPreTraining, ViTModel, ViTConfig
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Setup
warnings.filterwarnings('ignore')
setup_logger()

# Reproducibility
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
print(f"Environment Ready. Torch: {torch.__version__} ({'CUDA' if torch.cuda.is_available() else 'CPU'}), Detectron2: {detectron2.__version__}")

d:\Projects\dip\venv\Lib\site-packages\detectron2\model_zoo\model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Environment Ready. Torch: 2.8.0+cpu (CPU), Detectron2: 0.6


d:\Projects\dip\venv\Lib\site-packages\albumentations\__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
class Config:
    # Data paths
    DATA_ROOT = Path("data")
    UNLABELED_IMAGES_PATH = DATA_ROOT / "images"
    TRAIN_IMAGES_PATH = DATA_ROOT / "train" / "images"
    TRAIN_MASKS_PATH = DATA_ROOT / "train" / "masks"
    VAL_IMAGES_PATH = DATA_ROOT / "val" / "images"
    VAL_MASKS_PATH = DATA_ROOT / "val" / "masks"
    
    # Model paths
    MODELS_PATH = Path("models")
    SSL_BACKBONE_PATH = MODELS_PATH / "ssl_vit_backbone.pth"
    FINAL_MODEL_PATH = MODELS_PATH / "final_lesion_segmenter.pth"
    SSL_CONFIG_JSON = MODELS_PATH / "ssl_config.json"
    
    # Parameters
    IMAGE_SIZE = (224, 224)
    BATCH_SIZE_SSL = 16
    BATCH_SIZE_DETECTRON = 4
    NUM_EPOCHS_SSL = 20
    
    # Robust Device Check
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def __post_init__(self):
        self.MODELS_PATH.mkdir(exist_ok=True, parents=True)
        self.DATA_ROOT.mkdir(exist_ok=True, parents=True)

config = Config()
config.__post_init__()
print(f"Configuration loaded. Using Device: {config.DEVICE}")

Configuration loaded. Using Device: cpu


In [ ]:
class UnlabeledSkinDataset(Dataset):
    def __init__(self, data_dir):
        self.files = []
        for ext in ['*.jpg', '*.png', '*.jpeg']:
            self.files.extend(list(Path(data_dir).rglob(ext)))
        self.transform = transforms.Compose([
            transforms.Resize(256), transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(), transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        try: return self.transform(Image.open(self.files[idx]).convert('RGB'))
        except: return torch.zeros(3, 224, 224)

class MAETrainer:
    def __init__(self, config):
        self.config = config
        self.model = ViTMAEForPreTraining.from_pretrained('facebook/vit-mae-base').to(config.DEVICE)
        self.optimizer = optim.AdamW(self.model.parameters(), lr=1.5e-4, weight_decay=0.05)
    
    def train(self, dataloader):
        self.model.train()
        print(f"Starting SSL Training...")
        for epoch in range(self.config.NUM_EPOCHS_SSL):
            total_loss = 0
            for batch in dataloader:
                loss = self.model(pixel_values=batch.to(self.config.DEVICE)).loss
                self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()
                total_loss += loss.item()
            print(f"Epoch {epoch+1}: Loss {total_loss/len(dataloader):.4f}")

    def save_backbone(self):
        torch.save(self.model.vit.state_dict(), self.config.SSL_BACKBONE_PATH)
        c = {'image_size': (224,224), 'patch_size': 16, 'hidden_size': 768, 
             'num_hidden_layers': 12, 'num_attention_heads': 12, 'intermediate_size': 3072}
        with open(self.config.SSL_CONFIG_JSON, 'w') as f: json.dump(c, f)
        print("SSL Backbone saved.")

def run_ssl_training():
    if config.UNLABELED_IMAGES_PATH.exists() and list(config.UNLABELED_IMAGES_PATH.iterdir()):
        ds = UnlabeledSkinDataset(config.UNLABELED_IMAGES_PATH)
        MAETrainer(config).train(DataLoader(ds, batch_size=config.BATCH_SIZE_SSL, shuffle=True))
        MAETrainer(config).save_backbone()
    else:
        print("Skipping SSL: No unlabeled data found.")

run_ssl_training() 

Starting SSL Training...
Epoch 1: Loss 0.0252
Epoch 2: Loss 0.1222
Epoch 3: Loss 0.0409
Epoch 4: Loss 0.0327
Epoch 5: Loss 0.0317
Epoch 6: Loss 0.0310
Epoch 7: Loss 0.0308


In [ ]:
def get_melanoma_dicts(img_dir, mask_dir):
    dataset_dicts = []
    img_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        img_files.extend(list(Path(img_dir).glob(ext)))
        img_files.extend(list(Path(img_dir).glob(ext.upper())))
    
    for idx, img_path in enumerate(img_files):
        record = {"file_name": str(img_path), "image_id": idx}
        image = cv2.imread(str(img_path))
        if image is None: continue
        record["height"], record["width"] = image.shape[:2]
        
        mask_path = Path(mask_dir) / (img_path.stem + "_mask" + img_path.suffix)
        if not mask_path.exists():
             for alt in ['.png', '.jpg']:
                 if (Path(mask_dir)/(img_path.stem+"_mask"+alt)).exists():
                     mask_path = Path(mask_dir)/(img_path.stem+"_mask"+alt); break
        
        if not mask_path.exists(): continue
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None: continue
        
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        objs = []
        for c in contours:
            if cv2.contourArea(c) < 50: continue
            poly = c.flatten().tolist()
            if len(poly) < 6: continue
            x, y, w, h = cv2.boundingRect(c)
            objs.append({"bbox": [x, y, x+w, y+h], "bbox_mode": BoxMode.XYXY_ABS, 
                         "segmentation": [poly], "category_id": 0})
        if objs:
            record["annotations"] = objs
            dataset_dicts.append(record)
    return dataset_dicts

for d in ["train", "val"]:
    DatasetCatalog.register(f"melanoma_{d}", lambda d=d: get_melanoma_dicts(
        config.TRAIN_IMAGES_PATH if d=="train" else config.VAL_IMAGES_PATH,
        config.TRAIN_MASKS_PATH if d=="train" else config.VAL_MASKS_PATH))
    MetadataCatalog.get(f"melanoma_{d}").set(thing_classes=["lesion"])
print("Datasets registered.")

In [ ]:
class AlbumentationsMapper:
    def __init__(self, is_train=True):
        self.is_train = is_train
        self.aug = A.Compose([
            A.Resize(512, 512),                  # <---- CRITICAL FIX
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

        self.val_aug = A.Compose([
            A.Resize(512, 512),                 # <---- must match train size
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])


    def __call__(self, dataset_dict):
        d = dataset_dict.copy()
        image = cv2.cvtColor(utils.read_image(d["file_name"], format="BGR"), cv2.COLOR_BGR2RGB)
        
        masks, bboxes, labels = [], [], []
        if "annotations" in d:
            for anno in d["annotations"]:
                m = np.zeros((d["height"], d["width"]), dtype=np.uint8)
                for seg in anno["segmentation"]: cv2.fillPoly(m, [np.array(seg).reshape(-1, 2)], 1)
                masks.append(m); bboxes.append(anno["bbox"]); labels.append(anno["category_id"])
        
        try:
            if self.is_train:
                t = self.aug(image=image, masks=masks, bboxes=bboxes, class_labels=labels)
                image, masks, bboxes, labels = t['image'], t['masks'], t['bboxes'], t['class_labels']
            else:
                image = self.val_aug(image=image)['image']
        except: 
            image = self.val_aug(image=image)['image']
            masks, bboxes, labels = [], [], []

        # Get actual image dimensions after augmentation
        if isinstance(image, torch.Tensor):
            img_h, img_w = image.shape[1], image.shape[2]
        else:
            img_h, img_w = image.shape[:2]
        
        # Ensure bboxes, masks, and labels have the same length after augmentation
        # This can happen if augmentation filters some annotations
        min_len = min(len(bboxes), len(masks), len(labels))
        if min_len > 0:
            bboxes = bboxes[:min_len]
            masks = masks[:min_len]
            labels = labels[:min_len]
        else:
            bboxes, masks, labels = [], [], []

        target = Instances((img_h, img_w))
        if len(bboxes) > 0:
            target.gt_boxes = Boxes(torch.tensor(bboxes, dtype=torch.float32))
            target.gt_classes = torch.tensor(labels, dtype=torch.int64)
            target.gt_masks = BitMasks(torch.stack([torch.as_tensor(m, dtype=torch.bool) for m in masks]))
        else:
            target.gt_boxes = Boxes(torch.zeros((0, 4)))
            target.gt_classes = torch.zeros(0, dtype=torch.int64)
            target.gt_masks = BitMasks(torch.zeros((0, img_h, img_w)))
            
        d["image"] = image
        d["instances"] = target
        return d

In [ ]:
class ViTBackbone(Backbone):
    def __init__(self, vit_model):
        super().__init__()
        self.vit = vit_model
        self._out_features = ["res2", "res3", "res4", "res5"]
        self._out_feature_strides = {"res2": 4, "res3": 8, "res4": 16, "res5": 32}
        self._out_feature_channels = {k: 256 for k in self._out_features}
        
        self.projections = nn.ModuleList([nn.Conv2d(768, 256, 1) for _ in range(4)])
        
        # Upsampling for FPN compatibility
        self.up_res2 = nn.Sequential(
            nn.ConvTranspose2d(256, 256, 2, 2), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 256, 2, 2))
        self.up_res3 = nn.ConvTranspose2d(256, 256, 2, 2)
        self.identity = nn.Identity()
        self.down_res5 = nn.MaxPool2d(2, 2)
        
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)): weight_init.c2_msra_fill(m)
        
    def forward(self, x):
        B, _, H, W = x.shape
        
        # --- CRITICAL FIX: Bypass size check by using projection directly ---
        # Standard self.vit.embeddings.patch_embeddings(x) enforces 224x224.
        # We do manual projection and flattening to support variable sizes.
        x_p = self.vit.embeddings.patch_embeddings.projection(x) # (B, 768, H/16, W/16)
        x_p = x_p.flatten(2).transpose(1, 2) # (B, N, 768)
        
        cls = self.vit.embeddings.cls_token.expand(B, -1, -1)
        
        pos = self.vit.embeddings.position_embeddings
        grid_h, grid_w = H // 16, W // 16
        orig_size = int((pos.shape[1]-1)**0.5)
        pos_grid = torch.nn.functional.interpolate(
            pos[:, 1:].reshape(1, orig_size, orig_size, -1).permute(0,3,1,2),
            size=(grid_h, grid_w), mode='bilinear', align_corners=False
        ).permute(0,2,3,1).reshape(1, -1, 768)
        
        hidden = torch.cat((cls, x_p + pos_grid), dim=1)
        feats = []
        indices = [3, 6, 9, 12]
        
        for i, block in enumerate(self.vit.encoder.layer):
            hidden = block(hidden)[0]
            if i+1 in indices:
                out = hidden[:, 1:].reshape(B, grid_h, grid_w, 768).permute(0, 3, 1, 2)
                feats.append(self.projections[indices.index(i+1)](out))
                
        return {
            "res2": self.up_res2(feats[0]),
            "res3": self.up_res3(feats[1]),
            "res4": self.identity(feats[2]),
            "res5": self.down_res5(feats[3])
        }

@BACKBONE_REGISTRY.register()
def build_vit_backbone(cfg, input_shape):
    if config.SSL_BACKBONE_PATH.exists():
        with open(config.SSL_CONFIG_JSON) as f: c = json.load(f)
        vm = ViTModel(ViTConfig(**c, add_pooling_layer=False))
        vm.load_state_dict(torch.load(config.SSL_BACKBONE_PATH, map_location='cpu'), strict=False)
    else:
        vm = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k', add_pooling_layer=False)
    
    return FPN(
        bottom_up=ViTBackbone(vm),
        in_features=["res2", "res3", "res4", "res5"],
        out_channels=256, norm="", top_block=LastLevelMaxPool(), fuse_type="sum"
    )

In [ ]:
from detectron2.engine import DefaultTrainer

class AugTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=AlbumentationsMapper(True))


def run_mask_rcnn_training():

    if not any(config.TRAIN_IMAGES_PATH.iterdir()):
        print("No data found.")
        return

    # ensure backbone registry consistency
    if "build_vit_backbone" in BACKBONE_REGISTRY._obj_map:
        del BACKBONE_REGISTRY._obj_map["build_vit_backbone"]
    BACKBONE_REGISTRY.register(build_vit_backbone)

    cfg = get_cfg()
    cfg.merge_from_file(
        model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
    )

    # *** FORCED OVERRIDE *** (must be before trainer creation)
    cfg.MODEL.WEIGHTS = None
    cfg.MODEL.WEIGHTS = ""
    cfg.MODEL.WEIGHTS = " "   # multiple overrides to ensure no R-50 loads

    cfg.MODEL.BACKBONE.NAME = "build_vit_backbone"
    cfg.MODEL.BACKBONE.FREEZE_AT = 0
    cfg.MODEL.FPN.IN_FEATURES = ["res2", "res3", "res4", "res5"]

    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg.DATASETS.TRAIN = ("melanoma_train",)
    cfg.DATASETS.TEST = ("melanoma_val",)

    cfg.SOLVER.IMS_PER_BATCH = config.BATCH_SIZE_DETECTRON
    cfg.SOLVER.MAX_ITER = 500
    cfg.DATALOADER.NUM_WORKERS = 0

    cfg.MODEL.DEVICE = str(config.DEVICE)
    cfg.OUTPUT_DIR = str(config.MODELS_PATH / "detectron2_output")
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

    # *** CRITICAL *** ensure no checkpoints load
    trainer = AugTrainer(cfg)
    trainer.resume_or_load(resume=False)

    print("Training starting...")

    trainer.train()

    DetectionCheckpointer(trainer.model, save_dir=cfg.OUTPUT_DIR).save("final_model")

    shutil.copy(
        Path(cfg.OUTPUT_DIR) / "final_model.pth",
        config.FINAL_MODEL_PATH
    )

    print("Training Success!")


run_mask_rcnn_training()

In [ ]:
def evaluate():
    if not config.FINAL_MODEL_PATH.exists(): return
    
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.DEVICE = str(config.DEVICE)
    cfg.MODEL.BACKBONE.NAME = "build_vit_backbone"
    cfg.MODEL.FPN.IN_FEATURES = ["res2", "res3", "res4", "res5"]
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg.MODEL.PIXEL_MEAN = [0,0,0]; cfg.MODEL.PIXEL_STD = [1,1,1]
    
    model = build_model(cfg)
    DetectionCheckpointer(model).load(str(config.FINAL_MODEL_PATH))
    model.eval()
    
    dicts = get_melanoma_dicts(config.VAL_IMAGES_PATH, config.VAL_MASKS_PATH)
    for d in dicts[:3]:
        img = cv2.cvtColor(cv2.imread(d["file_name"]), cv2.COLOR_BGR2RGB)
        t = A.Compose([A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()])
        inp = t(image=img)["image"].unsqueeze(0).to(config.DEVICE)
        
        with torch.no_grad(): out = model([{"image": inp.squeeze(0)}])[0]
        
        v = Visualizer(img, metadata=MetadataCatalog.get("melanoma_val"))
        res = v.draw_instance_predictions(out["instances"].to("cpu"))
        plt.figure(figsize=(8,8)); plt.imshow(res.get_image()); plt.axis('off'); plt.show()

evaluate()